# UCS420 Assignment 7 - NLP-I
**Name:** Jaskaran Singh | **Roll no:** 1024170089 | **Batch:** 3Q13

Run the starter notebook cells FIRST (generate_dataset, process_ticket, TEST_LINES, tickets), then run these cells below them.

In [ ]:
import re, string, math
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nltk
for p in ['punkt','punkt_tab','stopwords','wordnet','omw-1.4',
          'averaged_perceptron_tagger','averaged_perceptron_tagger_eng']:
    nltk.download(p, quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag, bigrams
pd.set_option('display.max_rows', 500, 'display.width', 200)

# ---- EDIT THESE 2 LINES to match the starter notebook ----
TICKETS = tickets            # list of raw ticket strings
KEYS = dict(tokens='tokens', punct='punctuation', stop='stopwords', final='final_tokens')
# ----------------------------------------------------------
sample = process_ticket(TICKETS[0])
print(type(sample)); print(sample)   # check that KEYS match what is printed here

## Q1. Preprocessing

In [ ]:
def unpack(res):
    if isinstance(res, dict):
        return [res[KEYS[k]] for k in ('tokens','punct','stop','final')]
    return list(res)[:4]   # if it returns a tuple: tokens, punct, stopwords, final

results, rows = [], []
for i, raw in enumerate(TICKETS):
    tok, pun, stp, fin = unpack(process_ticket(raw))
    results.append((tok, pun, stp, fin))
    rows.append({'Ticket': i,
                 'Sentences': len(sent_tokenize(raw)),
                 'Tokens': len(tok),
                 'Punctuation': len(pun),
                 'Stopwords': len(stp),
                 'Unique tokens': len(set(t.lower() for t in tok)),
                 'Final tokens': len(fin)})
df1 = pd.DataFrame(rows)
total = df1.drop(columns='Ticket').sum(); total['Ticket'] = 'TOTAL'
df1 = pd.concat([df1, total.to_frame().T], ignore_index=True)
display(df1)

SEL = 0   # selected ticket
tok, pun, stp, fin = results[SEL]
print("Ticket:", TICKETS[SEL]); print("\nTokens:", tok)
print("\nPunctuation:", pun); print("\nStopwords:", stp); print("\nFinal tokens:", fin)

## Q2. Regex extraction and replacement

In [ ]:
P_LONG  = r"\b[A-Za-z]{6,}\b"                 # words longer than 5 letters
P_NUM   = r"\b\d+(?:\.\d+)*\b"                # 42, 95.5, 2.0.1 as single matches
P_CAP   = r"\b[A-Z][a-z]+\b"                  # capitalized words
P_VOWEL = r"\b[AEIOUaeiou][A-Za-z]*\b"        # words beginning with a vowel

P_EMAIL = r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"
P_URL   = r"(?:https?://|www\.)[^\s<>\"]*[^\s<>\".,;:!?)]"
P_PHONE = r"(?<![\w.])(?:\+?\d{1,3}[\s.-]?)?(?:\(\d{3}\)|\d{3})[\s.-]?\d{3}[\s.-]?\d{4}(?!\d)"

def extract(text):
    return {'long>5': re.findall(P_LONG, text), 'numbers': re.findall(P_NUM, text),
            'capitalized': re.findall(P_CAP, text), 'vowel-start': re.findall(P_VOWEL, text)}

def mask(text):
    text = re.sub(P_EMAIL, '<EMAIL>', text)
    text = re.sub(P_URL, '<URL>', text)
    return re.sub(P_PHONE, '<PHONE>', text)

print("=== TEST_LINES ===")
for line in TEST_LINES:
    print("\nRAW   :", line); print("MASKED:", mask(line))
    for k, v in extract(line).items(): print(f"  {k:12}: {v}")

print("\n=== TICKETS ===")
for i, raw in enumerate(TICKETS):
    print(f"\n[{i}] {mask(raw)}")
    for k, v in extract(raw).items(): print(f"  {k:12}: {v}")

## Q3. Custom tokenization (one RegexpTokenizer pattern)

In [ ]:
TOKEN_PATTERN = r"""(?x)
    [\w.+-]+@[\w-]+(?:\.[\w-]+)+                                   # email
  | (?:https?://|www\.)[^\s<>"]*[^\s<>".,;:!?)]                    # URL
  | (?:\+?\d{1,3}[\s.-]?)?(?:\(\d{3}\)|\d{3})[\s.-]\d{3}[\s.-]\d{4} # phone
  | [vV]?\d+(?:\.\d+)+                                             # decimals / versions
  | \w+(?:[-'\u2019]\w+)*                                          # words, Wi-Fi, isn't
  | [^\w\s]                                                        # any other punctuation
"""
_tk = RegexpTokenizer(TOKEN_PATTERN)
def my_tokenize(text):
    return _tk.tokenize(text)

diff_rows = []
for src_name, texts in (('TEST_LINES', TEST_LINES), ('ticket', TICKETS)):
    for j, t in enumerate(texts):
        a, b = my_tokenize(t), word_tokenize(t)
        if src_name == 'TEST_LINES':
            print("\nTEXT        :", t); print("my_tokenize :", a); print("word_tokenize:", b)
        if a != b:
            diff_rows.append({'source': f'{src_name}[{j}]',
                              'only my_tokenize': [x for x in a if x not in b],
                              'only word_tokenize': [x for x in b if x not in a]})
display(pd.DataFrame(diff_rows))

**Differences (confirm each against the table above and keep the ones that appear in YOUR output):**
1. **Contractions:** `word_tokenize` follows Penn Treebank and splits `isn't` -> `is`, `n't`; `my_tokenize` keeps `isn't` as one token because the pattern allows an apostrophe between word characters.
2. **URLs:** `word_tokenize` breaks a URL such as `https://site.com/help` into `https`, `:`, `//site.com/help`; `my_tokenize` keeps the full URL as one token.
3. **Phone numbers:** `word_tokenize` splits `(555) 123-4567` into `(`, `555`, `)`, `123-4567`; `my_tokenize` keeps it as one token.
4. **Versions/decimals/emails:** `word_tokenize` may split or mishandle versions with a prefix like `v2.0.1` or emails with trailing punctuation; our dedicated alternatives keep them whole.
5. **Trailing punctuation:** my pattern emits every punctuation mark as its own token, while `word_tokenize` sometimes attaches a final period to an abbreviation.

## Q4. Stemming and lemmatization

In [ ]:
unique_final = sorted(set(t.lower() for r in results for t in r[3]))
print(len(unique_final), "unique final tokens")

# Help choosing a suffix: count endings in YOUR data
for suf in ['ing','ed','ly','ion','ers','es','s']:
    ws = [w for w in unique_final if w.endswith(suf)]
    print(f"{suf:5}{len(ws):3}  e.g. {ws[:6]}")

In [ ]:
SUFFIX = r"ing$|ed$"        # EDIT after looking at the counts above
porter, lancaster = PorterStemmer(), LancasterStemmer()
regexp = RegexpStemmer(SUFFIX, min=4)
lemm = WordNetLemmatizer()

def to_wn(tag):
    return {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}.get(tag[0], wordnet.NOUN)

tags = dict(pos_tag(unique_final))
rows = []
for w in unique_final:
    rows.append({'word': w, 'POS': tags[w], 'Porter': porter.stem(w), 'Lancaster': lancaster.stem(w),
                 'Regexp': regexp.stem(w), 'Lemma(POS)': lemm.lemmatize(w, to_wn(tags[w]))})
df4 = pd.DataFrame(rows)
diff4 = df4[df4[['Porter','Lancaster','Regexp','Lemma(POS)']].nunique(axis=1) > 1]
display(diff4)

**Regexp suffix justification:** I chose `ing$|ed$` because my dataset contains words ending in these suffixes (see the counts above, e.g. two or more words such as ones printed in the `ing` and `ed` rows), and removing them maps inflected verbs to a base form. `min=4` stops short words (e.g. `red`, `bed`) from being damaged. *(Replace the example words with 2 real words from your output.)*

## Q5. Corpus statistics

In [ ]:
S1 = [t.lower() for r in results for t in r[0]]
S2 = [t for t in S1 if not all(c in string.punctuation for c in t)]
S3 = [t.lower() for r in results for t in r[3]]
S4 = [porter.stem(t) for t in S3]

def stats(name, s):
    n, u = len(s), len(set(s))
    return {'Set': name, 'Total tokens': n, 'Unique tokens': u, 'TTR': round(u / n, 4)}
display(pd.DataFrame([stats('S1 all', S1), stats('S2 no punct', S2),
                      stats('S3 final', S3), stats('S4 stemmed', S4)]))

print("Top 10 S1:", Counter(S1).most_common(10))
print("Top 10 S3:", Counter(S3).most_common(10))

# Zipf plot for S3
freq = sorted(Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(freq) + 1)
plt.figure(figsize=(6, 4))
plt.plot(np.log(ranks), np.log(freq), 'o-')
plt.xlabel('log(rank)'); plt.ylabel('log(frequency)'); plt.title('Zipf plot - S3 (final tokens)')
plt.grid(alpha=.3); plt.show()

# Bigrams (built within each ticket so they do not cross ticket boundaries)
def top_bigrams(token_lists, k=5):
    c = Counter()
    for toks in token_lists: c.update(bigrams(toks))
    return c.most_common(k)
print("Top 5 bigrams S1:", top_bigrams([[t.lower() for t in r[0]] for r in results]))
print("Top 5 bigrams S3:", top_bigrams([[t.lower() for t in r[3]] for r in results]))

## Q6. Interpretation
Helper cells below print real examples from your data; paste them into the answers.

In [ ]:
# (b) tokenizer difference
for t in TEST_LINES + TICKETS:
    a, b = my_tokenize(t), word_tokenize(t)
    if a != b:
        print("TEXT:", t); print("mine :", [x for x in a if x not in b]); print("nltk :", [x for x in b if x not in a]); break

# (c) bad stemming: different words sharing a stem, and non-word stems
groups = defaultdict(set)
for w in unique_final: groups[porter.stem(w)].add(w)
print("Collisions:", {s: ws for s, ws in groups.items() if len(ws) > 1})
print("Non-word stems:", [(w, porter.stem(w)) for w in unique_final
                          if porter.stem(w) not in unique_final and not wordnet.synsets(porter.stem(w))][:10])

# (d) lemma better than stem
print(df4[(df4['Lemma(POS)'] != df4['Porter']) & (df4['Lemma(POS)'] != df4['word'])][['word','POS','Porter','Lemma(POS)']].head(10))

**(a)** TTR = unique/total. S1 -> S2: removing punctuation lowers the total but barely changes the unique count, so TTR rises. S2 -> S3: stopwords (very frequent, repeated) are removed, so total drops a lot while unique drops less, TTR rises further. S3 -> S4: stemming merges inflected forms (e.g. *connect/connection/connecting*), reducing unique tokens while the total stays the same, so TTR falls (it is the lowest-vocabulary view of the same tokens).

**(b)** *(Use the example printed above.)* e.g. `isn't`: `word_tokenize` gives `is` + `n't` (Treebank rule that splits clitics), while `my_tokenize` keeps `isn't` whole because the pattern allows an internal apostrophe. Likewise a URL/phone number stays one token in mine but is split by `word_tokenize`.

**(c)** *(Use a printed collision / non-word stem.)* e.g. Porter turns `connection`/`connected` -> `connect` fine, but `university`/`universe` -> `univers` (different meanings merged) and `policy`/`police` -> `polici` (non-words / merged). Stems are not dictionary words, which hurts readability and can conflate unrelated words.

**(d)** *(Use a printed row.)* e.g. `better` tagged as ADJ lemmatizes to `good` and `was` tagged as VERB lemmatizes to `be`, while Porter leaves `better` or gives `wa`. `meeting` as a NOUN stays `meeting`, but as a VERB becomes `meet`; Porter always gives `meet`, losing the noun sense. POS information lets the lemmatizer keep a real word with the right meaning.